# Compare the same predictions using three annotation references

**Use a CPU runtime and Run all.** No GPU, training or new model inference
is needed. This notebook answers: how does the original supplied model
compare with the current two-view method when the evaluation annotations
change? It uses the exact predictions from the already completed common-
runtime inference comparison, and checks every input before scoring.

The references are the original supplied annotations (664 fronts), an
earlier fixed reference (666 fronts), and the reviewed reference (654
fronts). They describe the same 20 photographs. Every comparison uses
both methods on the same reference. Changing the reference changes the
measurement, not what the model predicted or learned.


## 1 Load the fixed implementation

Source and results are kept separately. If the existing source has been
edited, choose a new `BAOMAI_REFERENCE_MATRIX_BASE` directory.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "8ee1d751817638d72eb7b385a7f4164478a2b7e8"
DRIVER_SHA256 = "2939b2789abf2566d7680de60a0a87efae2a41e45c2dc01c9d15d1221cb8bbf1"
BASE = Path(os.environ.get("BAOMAI_REFERENCE_MATRIX_BASE", "/content/baomai-annotation-reference-r60")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Nonempty source preserved; choose a fresh BAOMAI_REFERENCE_MATRIX_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", *['/scripts/', '/data/manifest.json', '/data/labels/val/', '/data/references/sequence-refined-reference-2026-11-18/', '/data/reviews/fronts-2026-11-27-r50/', '/docs/thesis/evidence/original-baseline/fresh/baomai-original-comparison-r59-results.zip']], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
    raise RuntimeError("Changed source preserved; choose a fresh BAOMAI_REFERENCE_MATRIX_BASE.")
run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
run(["git", "sparse-checkout", "set", *['/scripts/', '/data/manifest.json', '/data/labels/val/', '/data/references/sequence-refined-reference-2026-11-18/', '/data/reviews/fronts-2026-11-27-r50/', '/docs/thesis/evidence/original-baseline/fresh/baomai-original-comparison-r59-results.zip']], SOURCE)
SCRIPT = SOURCE / "scripts/run_reference_matrix_r60.py"
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == DRIVER_SHA256
def command(action):
    return [sys.executable, "-B", SCRIPT, action, "--source-commit", SOURCE_COMMIT,
            "--script-sha256", DRIVER_SHA256, "--root", SOURCE, "--work", WORK]
print("Published implementation:", SOURCE_COMMIT)


hint: Using 'master' as the name for the initial branch. This default branch name
hint: will change to "main" in Git 3.0. To configure the initial branch name
hint: to use in all of your new repositories, which will suppress this warning,
hint: call:
hint:
hint: 	git config --global init.defaultBranch <name>
hint:
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint:
hint: 	git branch -m <name>
hint:
hint: Disable this message with "git config set advice.defaultBranchName false"
Initialized empty Git repository in /workspace/scratch/b575529246b5/reference-matrix-r60-execution/source/.git/
From https://github.com/Siuuuers/dl
 * branch            8ee1d751817638d72eb7b385a7f4164478a2b7e8 -> FETCH_HEAD
HEAD is now at 8ee1d75 Add supplied-annotation comparison with unchanged model predictions
Published implementation: 8ee1d751817638d72eb7b385a7f4164478a2b7e8


## 2 Verify the saved predictions and annotation references

The original annotation manifest is byte-identical to the manifest at
the historical upstream baomai revision. All 20 original label hashes
are verified. The two later references retain their own manifests.
The completed inference archive, inventory, model identities, photograph
hashes, suppression and coordinate averages are checked without loading
any model. No annotation is edited.


In [2]:
run(command("prepare"))
preflight = json.loads((WORK / "preflight.json").read_text())
print("Photographs:", len(preflight["contract"]["photographs"]))
print("References:", list(preflight["contract"]["references"]))
print(preflight["requirements"])


Verified unchanged predictions and three annotation references for all 20 photographs.
Photographs: 20
References: ['frozen666', 'r50', 'supplied664']
Python standard library only; no GPU, model execution, training or image downloads.


## 3 Recompute the complete comparison

All methods use object confidence strictly above 0.25, corner confidence
at least 0.50, matching intersection-over-union at least 0.50, and the same
additional suppression at overlap above 0.70. All eligible complete
reference objects remain in the five-/ten-pixel denominator, including
missed objects. Duplicates are included in extra predictions.

The previously reported fixed and reviewed scores must reproduce
exactly. The new supplied-reference row uses the same predictions and
scoring rule. References contain different entries and coordinates, so
a change between references is not automatically a change on identical
physical objects or evidence of training improvement.


In [3]:
run(command("run"))
summary = json.loads((WORK / "summary.json").read_text())
print("Annotation reference | method | misses | extras | duplicates | pairs within 5 pixels | pairs within 10 pixels")
for row in summary["matrix"]:
    print(row["reference_name"], row["method"], row["missed"], row["extras"], row["duplicates"],
          f"{row['within5']}/{row['eligible_pairs']}", f"{row['within10']}/{row['eligible_pairs']}", sep=" | ")
print(summary["interpretation"])


supplied664 original missed 39 extras 56 duplicates 11 pairs5 521/584 pairs10 568/584
supplied664 proposed missed 46 extras 69 duplicates 8 pairs5 514/584 pairs10 565/584
frozen666 original missed 26 extras 41 duplicates 11 pairs5 534/584 pairs10 580/584
frozen666 proposed missed 34 extras 55 duplicates 9 pairs5 525/584 pairs10 574/584
r50 original missed 34 extras 61 duplicates 5 pairs5 555/583 pairs10 577/583
r50 proposed missed 18 extras 51 duplicates 5 pairs5 569/583 pairs10 583/583
Result: /workspace/scratch/b575529246b5/reference-matrix-r60-execution/work/baomai-annotation-reference-r60-results.zip
Annotation reference | method | misses | extras | duplicates | pairs within 5 pixels | pairs within 10 pixels
Supplied annotations | original | 39 | 56 | 11 | 521/584 | 568/584
Supplied annotations | proposed | 46 | 69 | 8 | 514/584 | 565/584
Earlier fixed annotations | original | 26 | 41 | 11 | 534/584 | 580/584
Earlier fixed annotations | proposed | 34 | 55 | 9 | 525/584 | 574/584
Re

## 4 Verify and download the evidence

Verification recomputes every score, match, paired change and coordinate
average from the verified source inputs. The result ZIP preserves all
reference labels, predictions, per-image scores and individual outcomes.
Keep this ZIP if returning or comparing the analysis later.


In [4]:
run(command("verify"))
receipt = json.loads((WORK / "verification.json").read_text())
print("Verification:", receipt["status"], "Matrix cells:", receipt["matrix_cells"])
ARCHIVE = WORK / "baomai-annotation-reference-r60-results.zip"
assert ARCHIVE.is_file()
from IPython.display import FileLink, display
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(ARCHIVE)))
else:
    files.download(str(ARCHIVE))


{"archive_sha256": "ce8df4bcf223b6bee49fa67e829a443acb572e4a6aad75fa0f92657a16cc2601", "fresh_inference": false, "labels_modified": false, "matrix_cells": 6, "photographs": 20, "reference_count": 3, "status": "passed", "training": false}
Verification: passed Matrix cells: 6
/workspace/scratch/b575529246b5/reference-matrix-r60-execution/work/baomai-annotation-reference-r60-results.zip


## What would establish an effect from training annotations?

That is a different experiment: train two copies from the same starting
checkpoint on the same photographs, with the same settings and budget,
changing only the training labels. Evaluate both using one fixed
reference, then retain alternative-reference checks separately.

A historical 137-photograph experiment already made this comparison for
an earlier small annotation update. It does not isolate the full effect
of the latest labels, five additional photographs and current method.
Repeating inference alone cannot establish that causal training effect.

These are development photographs used during annotation review and
method selection. The matrix is not an independent capture evaluation,
and no model is adopted automatically.
